# Instructions

Notebook to run statistical testing for paper titled *A limited technical background is sufficient for attack-defense tree acceptability* accepted to 34th USENIX Security Symposium. The file necessary to run this notebook is `Survey Data.csv`. If this file is located in a different location to the jupyter notebook, you must change the `PATH` variable. 

The cells in this notebook are designed to be run in order. Once the `PATH` variable is set, the notebook can be run by Running All Cells. 

# Set up

## Set Up Environment

In [1]:
%pip install pingouin
%pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pingouin]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
import math
import pingouin as pg
import scipy
from scipy.stats import brunnermunzel

In [3]:
# Global Variables
HT_NAME = "LT"
LT_NAME = "HT"
# Path to folder hosting the .csv. Notebook assumes that provided .csv files are in the same folder as the notebook. 
PATH = ""




## Importing data

In [4]:
main_dataframe = pd.read_csv(PATH + "Survey Data.csv")
data_top = main_dataframe.head() 
main_dataframe

,Type,#,Programming,Threat Modeling Lecture Attendance,Have you heard of threat models before? If so which?,LS-ADT1-L1,LS-ADT1-L2,LS-ADT1-L3,LS-ADT1-L4,LS-ADT1-L5,...,SS-Q13,SS-Q14,SS-Q15,SS-Q16,SS-Q17,SS-Q17 Correct,SS-Q17 Reasoning,SS-Q18,SS-Q19,SS-Q20
0,LT,1,NaN,NaN,NaN,1.0,1.0,2.0,4.0,2.0,...,12.0,4.0,2.0,1.0,I don't get what the overall goal is,0.0,0.0,7.0,4.0,1.0
1,LT,2,0,Yes,Fault Trees and Event Trees,1.0,2.0,2.0,2.0,3.0,...,5.0,4.0,2.0,2.0,The overall goal is kept because all potential...,0.0,0.0,5.0,3.0,2.0
2,LT,3,2-3 months,No,No,2.0,3.0,3.0,2.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,LT,4,NaN,NaN,NaN,1.0,3.0,4.0,2.0,5.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LT,5,0,Yes,Fault Trees,1.0,2.0,2.0,3.0,1.0,...,6.0,5.0,1.0,2.0,No because some attack leaf nodes don't have d...,1.0,1.0,8.0,2.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,HT,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.0,4.0,2.0,2.0,yes,0.0,0.0,5.0,2.0,2.0
98,HT,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,HT,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,7.0,4.0,2.0,2.0,I do not know,0.0,0.0,6.0,4.0,3.0
100,HT,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,11.0,2.0,5.0,4.0,Yes since the entire attack tree focusses on m...,0.0,0.0,0.0,1.0,1.0


## Processing Data

The data artifact only contains participant responses. For hypothesis 1, we need to compare this against the expected responses. 

In [5]:
answers = {'ADT1 # def leaf nodes':2, 
        'ADT1 # atk leaf nodes':9, 
        'ADT1 Multi parent nodes':0,
        'ADT1 Multi refinement':0, 
        'ADT1 single child (atk)':0,
        'ADT1 multi countermeasure':0, 
        'ADT1 error count':0,
        'ADT1 # def nodes':3, 
        'ADT1 # atk nodes':15, 
        'ADT1 # and (atk)':2,
        'ADT1 #or (atk)':4, 
        'ADT1 LoA':4, 
        'ADT2 # def leaf nodes':3, 
        'ADT2 # def nodes':3,
        'ADT2 # atk leaf nodes':5, 
        'ADT2 # atk nodes':7, 
        'ADT2 # and (atk)':0,
        'ADT2 #or (atk)':2, 
        'ADT2 LoA':3, 
        'ADT2 Multi parent nodes':0,
        'ADT2 Multi refinement':0, 
        'ADT2 single child (atk)':0,
        'ADT2 multi countermeasure':0,
        'ADT2 # refinement':2, 
        'ADT2 error count':0,
        'ADT3 Multi parent nodes':0,
        'ADT3 Multi refinement':0, 
        'ADT3 single child (atk)':0,
        'ADT3 multi countermeasure':0, 
        'SS-Q2':3, 
        'SS-Q3':1, 
        'SS-Q4':2, 
        'SS-Q7':5,
        'SS-Q8':4, 
        'SS-Q9':0, 
        'SS-Q12':3, 
        'SS-Q13':6, 
        'SS-Q14':5,
        'SS-Q18':4}

def check_correct(ans, correct, margin):
    if(math.isnan(ans)):
        return math.nan
    if((ans + margin >= correct) and (ans - margin <= correct)):
        return 1
    else:
        return 0


def correctness(df, columns=[], margin=0, titleMargin=False):
    internalDF = df

    if len(columns) == 0:
        columns = answers.keys()
    for col in columns:
        title = "correct: " + col
        if titleMargin:
            title += " (margin: "+str(margin)+")"
        internalDF[title] = internalDF.apply(lambda row: check_correct(row[col], answers[col],margin),axis=1)
    
    return internalDF

# Correcting for h1
for test in ["SS-Q2", "SS-Q3", "SS-Q7", "SS-Q8", "SS-Q9", "SS-Q12", "SS-Q13", "SS-Q14", "SS-Q18", "ADT1 # atk leaf nodes", "ADT1 # def leaf nodes"]:
    main_dataframe = correctness(main_dataframe, [test])

# Correcting for h2-1
for test in ["ADT2 # def leaf nodes","ADT2 # def nodes","ADT2 # atk leaf nodes","ADT2 # atk nodes","ADT2 # and (atk)","ADT2 #or (atk)","ADT2 LoA"]:
    main_dataframe = correctness(main_dataframe, [test])

main_dataframe

,Type,#,Programming,Threat Modeling Lecture Attendance,Have you heard of threat models before? If so which?,LS-ADT1-L1,LS-ADT1-L2,LS-ADT1-L3,LS-ADT1-L4,LS-ADT1-L5,...,correct: SS-Q18,correct: ADT1 # atk leaf nodes,correct: ADT1 # def leaf nodes,correct: ADT2 # def leaf nodes,correct: ADT2 # def nodes,correct: ADT2 # atk leaf nodes,correct: ADT2 # atk nodes,correct: ADT2 # and (atk),correct: ADT2 #or (atk),correct: ADT2 LoA
0,LT,1,NaN,NaN,NaN,1.0,1.0,2.0,4.0,2.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0
1,LT,2,0,Yes,Fault Trees and Event Trees,1.0,2.0,2.0,2.0,3.0,...,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,LT,3,2-3 months,No,No,2.0,3.0,3.0,2.0,1.0,...,NaN,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
3,LT,4,NaN,NaN,NaN,1.0,3.0,4.0,2.0,5.0,...,NaN,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
4,LT,5,0,Yes,Fault Trees,1.0,2.0,2.0,3.0,1.0,...,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,HT,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,HT,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,HT,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100,HT,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
main_dataframe["leaf_node_avg"] = main_dataframe[["correct: SS-Q2", "correct: SS-Q7", "correct: ADT1 # atk leaf nodes"]].mean(axis=1)
main_dataframe["attack_vector_avg"] = main_dataframe[["correct: SS-Q8", "correct: SS-Q13"]].mean(axis=1)
main_dataframe["LoA_avg"] = main_dataframe[["correct: SS-Q14", "correct: SS-Q18"]].mean(axis=1)
main_dataframe["def_node_avg"] = main_dataframe[["correct: SS-Q9", "correct: SS-Q12", "SS-Q17 Correct", "correct: ADT1 # def leaf nodes"]].mean(axis=1)


## Setting Up Tests

### Brunner Munzel Test

Test for statistically significant difference

In [7]:
# Provide dataframe and column(s) in that dataframe, returns brunner munzel values for those columns according to the group names
def brunmun_test(df, columns=[]):
    testDF = pd.DataFrame()
    row = {}
    cols = columns
    if(len(cols)==0):
        cols = df.columns[2:]
    for col in cols:
        tempDF = df[col].dropna()
        ncsArray = np.array(tempDF.loc[df['Type'] == LT_NAME])
        csArray = np.array(tempDF.loc[df['Type'] == HT_NAME])
        test = brunnermunzel(ncsArray, csArray)
        row["column"] = col
        row["pvalue"] = float(test.pvalue)
        row["statistic"] = float(test.statistic)
        testDF = pd.concat([testDF, pd.DataFrame([row])],  ignore_index=True)
    testDF = testDF.sort_values(by=["pvalue"])
    return testDF

### TOST

Test for statistically significant equivalence. Applied in cases where no statistically significant difference is found.

In [8]:
def tost_test(df, columns=[]):
    testDF = pd.DataFrame()
    row = {}
    cols = columns
    if(len(cols)==0):
        cols = df.columns[2:]
    for col in cols:
        tempDF = df[col].dropna()
        ncsArray = np.array(tempDF.loc[df['Type'] == LT_NAME])
        csArray = np.array(tempDF.loc[df['Type'] == HT_NAME])
        test = pg.tost(ncsArray, csArray).loc['TOST', 'pval']
        row["column"] = col
        row["pvalue"] = float(test)
        testDF = pd.concat([testDF, pd.DataFrame([row])],  ignore_index=True)
    testDF = testDF.sort_values(by=["pvalue"])
    return testDF

### Cohen's d

Measure of effect size

In [9]:
def cohen_d(x,y):
        return (np.mean(x) - np.mean(y)) / math.sqrt((np.std(x, ddof=1) ** 2 + np.std(y, ddof=1) ** 2) / 2.0)

def cohen_test(df, columns = []):
    testDF = pd.DataFrame()
    row = {}
    cols = columns
    if(len(cols)==0):
        cols = df.columns[2:]
    for col in cols:
        tempDF = df[col].dropna()
        ncsArray = np.array(tempDF.loc[df['Type'] == LT_NAME])
        csArray = np.array(tempDF.loc[df['Type'] == HT_NAME])
        test = cohen_d(ncsArray, csArray)
        row["column"] = col
        row["cohen_d"] = test
        testDF = pd.concat([testDF, pd.DataFrame([row])],  ignore_index=True)
    testDF = testDF.sort_values(by=["cohen_d"])
    return testDF



# Running Statistical Tests

In [10]:
# Stores which columns are used to evaluate each hypothesis 
hypotheses = {
"h1": ["correct: SS-Q3", "leaf_node_avg", "def_node_avg", "attack_vector_avg", "LoA_avg"],
"h2_1": ["ADT2 # def leaf nodes","ADT2 # def nodes","ADT2 # atk leaf nodes","ADT2 # atk nodes","ADT2 # and (atk)","ADT2 #or (atk)","ADT2 LoA"],
"h2_2": ["ADT3 Cohesive","ADT3 Clear","ADT3 Concise","ADT3 Complete"],
"h3": ["ADT1 Multi parent nodes","ADT3 Multi parent nodes","ADT1 Multi refinement","ADT3 Multi refinement","ADT1 multi countermeasure","ADT2 multi countermeasure","ADT3 multi countermeasure","ADT1 single child (atk)","ADT2 single child (atk)","ADT3 single child (atk)"],
"h4":["LS-ADT1-L1", "SS-Q5", "SS-Q10", "SS-Q15", "SS-Q19"],
"h5" :["LS-ADT3-L3"],
"h6" :["LS-ADT1-L5", "LS-ADT2-L2", "LS-ADT3-L1"],
"h7" :["LS-ADT2-L1", "LS-ADT3-L2", "SS-Q6", "SS-Q11", "SS-Q16", "SS-Q20"],
"h8" :["LS-ADT3-W3 Yes", "LS-ADT3-W3 Communication", "LS-ADT3-W3 Analysis", "LS-ADT3-W5 Yes"],
"h9" :["ADT3 # def leaf nodes","ADT3 # def nodes","ADT3 # atk leaf nodes","ADT3 # atk nodes","ADT3 # and (atk)","ADT3 #or (atk)","ADT3 LoA", "ADT3 and:or ratio"],
}

## Corrections

Before compiling test results, we need to establish the corrections. This requires running all tests, and then multiplying by the number of tests applied in decreasing order. We use the Bonferonni-Holm method for correction. For simplicity of applying the correction and for displaying the results, instead of dividing the significant threshold by the correction value, we multiply the p-value by the correction value. This has the same effect as the correction value is always positive (a value ranging between number of tests applied and 1). After correcting the p-values, we will compile all results together.

In [226]:
# Set up data store
test_data = [["hypothesis", "test", "original p-value", "corrected p-value"]]

# Add grade data - for ethics reasons, the individual grade data cannot be released. The p-values are provided after running them separately.

test_data.append(["Grade: LT (all)", "BM", 0.06415, 0.06415])
test_data.append(["Grade: LT (all)", "TOST", 0.1741, 0.1741])
test_data.append(["Grade: HT (all)", "BM", 0.0366, 0.0366])
test_data.append(["Grade: HT (all)", "TOST", 0.01203, 0.01203])
test_data.append(["Grade: LT (SS)", "BM", 0.1027, 0.1027])
test_data.append(["Grade: LT (SS)", "TOST", 0.01623, 0.01623])
test_data.append(["Grade: HT (SS)", "BM", 0.2398, 0.2398])
test_data.append(["Grade: HT (SS)", "TOST", 0.034917, 0.034917])
test_data.append(["Grade: All", "BM", 0.1698, 0.1698])
test_data.append(["Grade: All", "TOST", 0.0004831533, 0.0004831533])

# Run Initial Tests
for hypothesis in hypotheses:
    for test in hypotheses[hypothesis]:
        bm_value = brunmun_test(main_dataframe, [test])
        test_data.append([test, "BM", float(bm_value["pvalue"].iloc[0]), float(bm_value["pvalue"].iloc[0])])
        # There is a glitch for LS-ADT3-L1, so this is manually added
        if float(bm_value["pvalue"].iloc[0]) > 0.05 or math.isnan(float(bm_value["pvalue"].iloc[0])):
            tost_val = tost_test(main_dataframe, [test])
            test_data.append([test, "TOST", float(tost_val["pvalue"].iloc[0]), float(tost_val["pvalue"].iloc[0])])
test_df = pd.DataFrame(test_data[1:], columns=test_data[0])
test_df = test_df.drop_duplicates()
test_df = test_df.sort_values(by=["original p-value"])

# Apply correction
def correct(x):
    global current_count
    current_count -= 1
    new_p_value = x * (current_count + 1)
    if new_p_value > 1:
        return 1.0
    else:
        return new_p_value 

corrections_needed = True

# Repeat corrections until all tests corrected
while corrections_needed:
    corrections_needed = False

    # Apply corrections
    current_count = len(test_df) 
    test_df = test_df.sort_values(by=["original p-value"])
    test_df["corrected p-value"] = test_df.apply(lambda x: correct(x["original p-value"]), axis=1)

    # Check if new tests are needed (due to a test that was statistically significant before correction, now we need to introduce a new TOST test
    for hypothesis in hypotheses:
        for test in hypotheses[hypothesis]:
            # Get test data from hypotheses
            tests = test_df[test_df["hypothesis"] == test]
            # If two tests already exist, skip this iteration
            if len(tests) == 2:
                continue
            if float(tests[tests["test"] == "BM"]["corrected p-value"].iloc[0]) > 0.05:
                corrections_needed = True
                new_test_df = [["hypothesis", "test", "original p-value", "corrected p-value"]]
                tost_val = tost_test(main_dataframe, [test])
                new_test_df.append([test, "TOST", float(tost_val["pvalue"].iloc[0]), float(tost_val["pvalue"].iloc[0])])
                new_test_df = pd.DataFrame(new_test_df[1:], columns=new_test_df[0])
                test_df = pd.concat([test_df, new_test_df])

# Apply again (unnecessary and shouldn't change result, but just in case)
current_count = len(test_df) 
test_df = test_df.sort_values(by=["original p-value"])
test_df["corrected p-value"] = test_df.apply(lambda x: correct(x["original p-value"]), axis=1)
                
test_df

/opt/homebrew/Cellar/jupyterlab/4.3.4_1/libexec/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
/opt/homebrew/Cellar/jupyterlab/4.3.4_1/libexec/lib/python3.13/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,hypothesis,test,original p-value,corrected p-value
31,ADT2 LoA,TOST,4.927302e-61,5.567851e-59
87,LS-ADT3-W3 Yes,TOST,3.558050e-42,3.985016e-40
47,ADT3 Multi refinement,TOST,3.188925e-28,3.539706e-26
28,ADT2 # and (atk),TOST,3.422969e-27,3.765266e-25
51,ADT2 multi countermeasure,TOST,9.048266e-24,9.862610e-22
...,...,...,...,...
90,LS-ADT3-W3 Analysis,BM,8.969684e-01,1.000000e+00
27,ADT2 # and (atk),BM,9.119751e-01,1.000000e+00
68,LS-ADT3-L3,BM,9.511257e-01,1.000000e+00
82,SS-Q16,BM,9.639867e-01,1.000000e+00


## Show Results

In [227]:
def standardize_length(x):
    end = 4 - int(len(x)/8)
    for _ in range(0, end, 1):
        x += "\t"
    return x

def float_to_exponential(num):
    return "{:e}".format(num)

hypotheses["G"] = ["Grade: LT (all)", "Grade: HT (all)", "Grade: LT (SS)", "Grade: HT (SS)", "Grade: All"]

for hypothesis in hypotheses:
    print("----- Hypothesis: " + hypothesis + " -----\n")
    print(standardize_length("Value") + standardize_length("BM Test")  + standardize_length(" ")  + standardize_length("TOST Test")  + standardize_length("Effect Size") )
    print(standardize_length(" ") + standardize_length("statistic")  + standardize_length("p-value")  + standardize_length("p-value")  + standardize_length("Cohen's d") )
    for test in hypotheses[hypothesis]:
        test_line = standardize_length(test)
        if test.split(" ")[0] != "Grade:":
            bm_value = brunmun_test(main_dataframe, [test])
            # print(bm_value)
            test_line += standardize_length(str(float(bm_value["statistic"].iloc[0])))
        else:
            test_line += standardize_length(" ") 
        tests = test_df[test_df["hypothesis"] == test]
        test_line += standardize_length(str(float_to_exponential(float(tests[tests["test"] == "BM"]["corrected p-value"].iloc[0]))))
        try:
            test_line += standardize_length(str(float_to_exponential(float(tests[tests["test"] == "TOST"]["corrected p-value"].iloc[0]))))
        except:
            test_line += standardize_length(" ") 
        if test.split(" ")[0] != "Grade:":
            test_line += standardize_length(str(round(float(cohen_test(main_dataframe, [test])["cohen_d"].iloc[0]), 2)))
        else:
            test_line += standardize_length(" ") 
        print(test_line)
    print("\n-----     -----\n\n\n")

----- Hypothesis: h1 -----

Value				BM Test				 				TOST Test			Effect Size			
 				statistic			p-value				p-value				Cohen's d			
correct: SS-Q3			0.2802136706214916		1.000000e+00			2.000900e-17			-0.07				
leaf_node_avg			-3.738531758382994		2.472131e-02			 				0.75				
def_node_avg			-1.178836307962709		1.000000e+00			1.478584e-19			0.25				
attack_vector_avg		-0.8407201295235008		1.000000e+00			6.185872e-10			0.24				
LoA_avg				-1.3629443099923284		1.000000e+00			8.809533e-17			0.38				

-----     -----



----- Hypothesis: h2_1 -----

Value				BM Test				 				TOST Test			Effect Size			
 				statistic			p-value				p-value				Cohen's d			
ADT2 # def leaf nodes		-0.6556553904234312		1.000000e+00			1.439466e-11			0.07				
ADT2 # def nodes		-1.2761231927471917		1.000000e+00			4.879236e-07			0.23				
ADT2 # atk leaf nodes		1.4347341050513354		1.000000e+00			1.729025e-16			-0.33				
ADT2 # atk nodes		1.72715614617854		1.000000e+00			3.525920e-04			-0.31				
ADT2 # and (atk)		0.11088782